# 事件质量规则的配对复核

这份 Notebook 用于**阅读和解释**已运行的实验结果；生成完整 CSV 和图的脚本是 `run_paired_event_quality_validation.py`。

所有方法在同一个场景、事件数、重复和位置误差组合下配对比较。增益为正表示比 `hard` 更好。请注意：重复抽样不等于独立探测器实验。

In [ ]:
from pathlib import Path
import pandas as pd

# VS Code 有时以项目根目录作为工作目录，有时以 Notebook 所在目录作为工作目录。
base = Path.cwd()
results = base / 'paired_event_quality_results'
if not results.exists():
    results = base / 'muon_analysis' / 'paired_event_quality_results'
if not results.exists():
    raise FileNotFoundError('请先运行 run_paired_event_quality_validation.py')
overall = pd.read_csv(results / 'paired_overall_noisy.csv')
conditions = pd.read_csv(results / 'paired_by_condition.csv')
overall[['method', 'paired_n', 'iou_gain_mean', 'iou_win_fraction',
         'global_error_gain_mm_mean', 'auto_error_gain_mm_mean']].round(4)

## 检查各场景，而不是只看总体平均

下面固定 5000 个输入事件。表中是每个场景与探测器误差下，相对 `hard` 的平均 IoU 增益。负数代表该方法反而降低 IoU。

In [ ]:
view = conditions[(conditions['event_count'] == 5000) &
                  (conditions['position_sigma_mm'] > 0)]
view.pivot_table(index=['scenario', 'position_sigma_mm'],
                 columns='method', values='iou_gain_mean').round(3)

## 读图提示

左图是 IoU 相对 `hard` 的变化，右图是自动窗口定位误差的减少量；两图的零线表示与 `hard` 持平。不能仅凭总体曲线宣布某个方法在全部场景最优。

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(results / 'paired_tradeoff_5000.png')))